In [ ]:
!sudo apt-get install tesseract-ocr
!sudo apt-get install tesseract-ocr-ara
!wget https://github.com/jishakrishnan/pytrsseract-arabic/blob/main/ara_num_test.traineddata -P /usr/share/tesseract-ocr/4.00/tessdata/
!tesseract --list-langs
!source ~/.bashrc
!pip3 install opencv-python
!pip3 install matplotlib
!pip3 install pytesseract
!pip install scipy
!pip install pytesserect
!sudo apt-get install tesseract-ocr
!sudo apt-get install tesseract-ocr-ara
!sudo apt-get install tesseract-ocr-ara # For Arabic language support

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 45 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 1s (3,312 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debc

In [ ]:
import cv2
import imutils
from pytesseract import image_to_string
import numpy as np
from datetime import datetime
import cv2
import imutils
from pytesseract import image_to_string
import numpy as np
from datetime import datetime
from google.colab.patches import cv2_imshow
import io
from PIL import Image
import cv2
import numpy as np
import imutils
import seaborn as sns
from pytesseract import image_to_string
import io
from PIL import Image

In [ ]:
def crop_text(img):
    # Apply thresholding
    ret, thresh1 = cv2.threshold(img, 0, 255, cv2.THRESH_OTSU | cv2.THRESH_BINARY_INV)
    rect_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 50))
    dilation = cv2.dilate(thresh1, rect_kernel, iterations=1)

    # Find contours
    contours, hierarchy = cv2.findContours(dilation, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    # Initialize variables to store the largest bounding box
    largest_area = 0
    largest_bbox = None

    # Loop through contours to find the largest bounding box
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h
        if area > largest_area:
            largest_area = area
            largest_bbox = (x, y, w, h)

    # If a bounding box was found, crop the image
    if largest_bbox is not None:
        x, y, w, h = largest_bbox
        cropped = img[y:y + h, x:x + w]
        return cropped
    else:
        return None

# Function to resize an image while maintaining aspect ratio
def image_resize(image, width=None, height=None, inter=cv2.INTER_AREA):
    dim = None
    (h, w) = image.shape[:2]

    if width is None and height is None:
        return image

    if width is None:
        r = height / float(h)
        dim = (int(w * r), height)
    else:
        r = width / float(w)
        dim = (width, int(h * r))

    resized = cv2.resize(image, dim, interpolation=inter)
    return resized

# Function to calculate corners of a contour
def calculate_corners(contour_points):
    corners = np.zeros((4, 2), dtype="float32")
    sum_points = contour_points.sum(axis=1)
    corners[0] = contour_points[np.argmin(sum_points)]
    corners[2] = contour_points[np.argmax(sum_points)]
    diff_points = np.diff(contour_points, axis=1)
    corners[1] = contour_points[np.argmin(diff_points)]
    corners[3] = contour_points[np.argmax(diff_points)]
    return corners

# Function to apply perspective transform to the image
def transform_perspective(input_image, contour_points):
    corners = calculate_corners(contour_points)
    (top_left, top_right, bottom_right, bottom_left) = corners

    width_top = np.linalg.norm(bottom_right - bottom_left)
    width_bottom = np.linalg.norm(top_right - top_left)
    max_width = max(int(width_top), int(width_bottom))

    height_left = np.linalg.norm(top_right - bottom_right)
    height_right = np.linalg.norm(top_left - bottom_left)
    max_height = max(int(height_left), int(height_right))

    destination = np.array([
        [0, 0],
        [max_width - 1, 0],
        [max_width - 1, max_height - 1],
        [0, max_height - 1]
    ], dtype="float32")

    matrix = cv2.getPerspectiveTransform(corners, destination)
    output_image = cv2.warpPerspective(input_image, matrix, (max_width, max_height))

    return output_image

# Function to process an image of an ID card
def process_image(image):
    original_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    image_ratio = original_image.shape[0] / 500.0
    resized_image = imutils.resize(original_image, height=500)

    gray_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2GRAY)
    blurred_image = cv2.GaussianBlur(gray_image, (5, 5), 0)
    edge_detected = cv2.Canny(blurred_image, 75, 200)

    # Find contours in the edge-detected image
    contours = cv2.findContours(edge_detected.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = imutils.grab_contours(contours)
    sorted_contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]

    detected_screen = None
    for contour in sorted_contours:
        contour_perimeter = cv2.arcLength(contour, True)
        approximated_contour = cv2.approxPolyDP(contour, 0.02 * contour_perimeter, True)
        if len(approximated_contour) == 4:
            detected_screen = approximated_contour
            break

    if detected_screen is not None:
        transformed_image = transform_perspective(original_image, detected_screen.reshape(4, 2) * image_ratio)
        transformed_image = image_resize(transformed_image, height=630)
        return transformed_image
    else:
        return None

# Function to convert Arabic-Indic numbers to normal numbers
def arabic_indic_to_normal(numbers):
    arabic_indic = '٠١٢٣٤٥٦٧٨٩'
    normal_numbers = '0123456789'
    translate_table = str.maketrans(arabic_indic, normal_numbers)
    return numbers.translate(translate_table)

# Function to remove unwanted characters from a string
def remove_new_lines(data):
    characters_to_remove = ['\n', ':', '    ', '   ', '  ', '،', '؛', '؟', ',', "'", '"', ';', '\x0c']
    for char in characters_to_remove:
        data = data.replace(char, '')
    return data

# Function to apply adaptive binarization to an image
def adaptive_binarization(img):
    _, _, img_gray = cv2.split(img)
    _, img_bw = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    filtered_image = cv2.bitwise_and(img, img, mask=img_bw)
    im_gray = cv2.cvtColor(filtered_image, cv2.COLOR_BGR2GRAY)
    _, im_bw = cv2.threshold(im_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    return im_bw

# Function to extract data from the front side of an ID card
def front_read(front_image):
    custom_config = r'-c tessedit_char_whitelist="٠١٢٣٤٥٦٧٨٩-ءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهويىپچڤگ " --psm 6'
    num_custom_config = r'-c tessedit_char_whitelist="٠١٢٣٤٥٦٧٨٩-ءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهويىپچڤگ " --psm 6'

    name_img_1 = crop_text(adaptive_binarization(front_image[150:230, 600:1000]))
    name_img_2 = crop_text(adaptive_binarization(front_image[220:310, 400:1000]))

    address_img_1 = crop_text(adaptive_binarization(front_image[280:370, 370:1000]))
    address_img_2 = crop_text(adaptive_binarization(front_image[360:450, 370:1000]))

    id_img = adaptive_binarization(front_image[470:580, 400:1000])

    name_1 = remove_new_lines(image_to_string(name_img_1, lang="ara", config=custom_config))
    name_2 = remove_new_lines(image_to_string(name_img_2, lang="ara", config=custom_config))

    address_1 = remove_new_lines(image_to_string(address_img_1, lang="ara", config=custom_config))
    address_2 = remove_new_lines(image_to_string(address_img_2, lang="ara", config=custom_config))

    name = name_1 + " " + name_2
    address = address_1 + " " + address_2

    id_number = remove_new_lines(image_to_string(id_img, lang="ara_num", config=num_custom_config))
    id_number = ''.join(id_number.split())
    normal_id_number = arabic_indic_to_normal(id_number)

    if normal_id_number[0] == '2':
        year = '19' + normal_id_number[1:3]
    else:
        year = '20' + normal_id_number[1:3]

    month = normal_id_number[3:5]
    day = normal_id_number[5:7]
    birth_date = f"{year}/{month}/{day}"
    gender_digit = int(normal_id_number[12:13])
    gender = 'male' if gender_digit % 2 else 'female'

    return {
        'name': name,
        'address': address,
        'id_number': normal_id_number,
        'birth_date': birth_date,
        'gender': gender
    }

# Function to process the ID card image and extract data
def process_id_card(image):
    transformed_image = process_image(image)
    if transformed_image is not None:
        return front_read(transformed_image)
    else:
        return None


In [ ]:


process_id_card('safey.jpeg')

error: OpenCV(4.10.0) :-1: error: (-5:Bad argument) in function 'cvtColor'
> Overload resolution failed:
>  - src data type = <U10 is not supported
>  - Expected Ptr<cv::UMat> for argument 'src'


In [ ]:

image = Image.open('Adham2.jpeg')
image = np.array(image)
process_id_card(image)


{'name': 'ادهم ',
 'address': '\u200e١\u200f فع \u200e١١\u200f فيلدج جاردن القطامية القطاميه - القاهره',
 'id_number': '30501090106256',
 'birth_date': '2005/01/09',
 'gender': 'male'}